# DistilBERT (3 seeds) vs. TF-IDF+SVM — Register Classification
## Methodological case study rebuilding part of Thesis Ch. 5 with proper rigor

**Framing for the write-up:** this is not a reproduction of the original BERT/XLNet
comparison. It is a focused, self-contained experiment on the same dataset,
designed to demonstrate methodology the original thesis lacked: a clean
train/val/test split with no test-set leakage, multiple training seeds with
mean ± std (not a single run treated as ground truth), explicit justification
for the macro-F1 vs. accuracy choice, and a real significance test (McNemar's)
for the headline comparison instead of an unexamined "tie."

**Before running:** upload `train_clean.csv`, `val_clean.csv`, `test_clean.csv`,
and `svm_test_predictions.csv` to the Colab working directory. Set Runtime >
Change runtime type > GPU (T4 is fine).

**Expect ~3x the runtime of a single-seed run** since DistilBERT trains 3 times
(seeds 42, 123, 2024). On a T4 with 4 epochs each, budget roughly 1.5–2 hours total.


In [ ]:
!pip install -q transformers datasets evaluate accelerate statsmodels


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEEDS = [42, 123, 2024]


## Load the cleaned, pre-split data

Rare classes (<20 train examples) already dropped upstream. Stratified train/val split. Test set is the original, untouched test file — never used for model selection.

In [ ]:
train_df = pd.read_csv('train_clean.csv')
val_df = pd.read_csv('val_clean.csv')
test_df = pd.read_csv('test_clean.csv')

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("Classes:", train_df['Label'].nunique())

label_encoder = LabelEncoder()
label_encoder.fit(train_df['Label'])
num_labels = len(label_encoder.classes_)
print("num_labels:", num_labels)

train_df['label_id'] = label_encoder.transform(train_df['Label'])
val_df['label_id'] = label_encoder.transform(val_df['Label'])
test_df['label_id'] = label_encoder.transform(test_df['Label'])


## Tokenization

Documents are long (mean ~1100 words). DistilBERT's max is 512 tokens, so most
documents get truncated. This is stated explicitly as a limitation in the
write-up, not hidden — it's a real constraint of using off-the-shelf BERT-family
models on long documents, and is exactly the kind of thing the original thesis
should have discussed but didn't.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
from torch.utils.data import Dataset

class RegisterDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]), truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = RegisterDataset(train_df['Text'], train_df['label_id'], tokenizer)
val_ds = RegisterDataset(val_df['Text'], val_df['label_id'], tokenizer)
test_ds = RegisterDataset(test_df['Text'], test_df['label_id'], tokenizer)

print(len(train_ds), len(val_ds), len(test_ds))


## Class weighting

Passed into the loss function (not resampling, to avoid duplicating/discarding long documents).

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_labels),
    y=train_df['label_id'].values
)
class_weights_t = torch.tensor(class_weights, dtype=torch.float)
print(class_weights_t)


## Train DistilBERT across 3 seeds

Each seed gets a fresh model initialization and a fresh Trainer. We collect
per-seed test predictions and metrics, then report mean ± std — this is the
direct fix for the original thesis's unexamined single-run "tie" claim.

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_t.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
    }


In [ ]:
seed_results = []          # per-seed summary metrics
seed_test_preds = {}       # seed -> array of predicted label strings (for McNemar / majority vote later)

for seed in SEEDS:
    print(f"\n========== SEED {seed} ==========")
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

    training_args = TrainingArguments(
        output_dir=f'./distilbert_register_seed{seed}',
        num_train_epochs=4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        report_to='none',
        seed=seed,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    test_output = trainer.predict(test_ds)
    test_preds_id = np.argmax(test_output.predictions, axis=-1)
    test_labels_id = test_output.label_ids

    test_preds = label_encoder.inverse_transform(test_preds_id)
    test_labels_str = label_encoder.inverse_transform(test_labels_id)

    acc = accuracy_score(test_labels_str, test_preds)
    macro_f1 = f1_score(test_labels_str, test_preds, average='macro', zero_division=0)
    print(f"Seed {seed} -> test accuracy: {acc:.4f}, test macro-F1: {macro_f1:.4f}")

    seed_results.append({'seed': seed, 'test_accuracy': float(acc), 'test_macro_f1': float(macro_f1)})
    seed_test_preds[seed] = test_preds

    # free GPU memory before next seed
    del model, trainer
    torch.cuda.empty_cache()

test_labels_str_final = test_labels_str  # same across seeds, same test set/order


## Aggregate across seeds: mean ± std

In [ ]:
seed_results_df = pd.DataFrame(seed_results)
print(seed_results_df)

acc_mean, acc_std = seed_results_df['test_accuracy'].mean(), seed_results_df['test_accuracy'].std()
f1_mean, f1_std = seed_results_df['test_macro_f1'].mean(), seed_results_df['test_macro_f1'].std()

print(f"\nDistilBERT test accuracy: {acc_mean:.4f} +/- {acc_std:.4f}  (n={len(SEEDS)} seeds)")
print(f"DistilBERT test macro-F1: {f1_mean:.4f} +/- {f1_std:.4f}  (n={len(SEEDS)} seeds)")

seed_results_df.to_csv('distilbert_seed_results.csv', index=False)


## Pick a representative seed run for the paired significance test

McNemar's test needs one fixed set of paired predictions per model. We use the
**median-macro-F1 seed** (not the best — using the best run would bias the
comparison in DistilBERT's favor) as DistilBERT's representative run against
the SVM baseline. The full 3-seed mean ± std is still reported as the headline
result; this representative run is only for the paired test.

In [ ]:
median_idx = seed_results_df['test_macro_f1'].sort_values().index[len(SEEDS) // 2]
representative_seed = int(seed_results_df.loc[median_idx, 'seed'])
print(f"Representative seed for McNemar's test: {representative_seed}")

bert_test_preds = seed_test_preds[representative_seed]

bert_pred_df = pd.DataFrame({
    'text': test_df['Text'].values,
    'y_true': test_labels_str_final,
    'y_pred': bert_test_preds,
})
bert_pred_df.to_csv('bert_test_predictions.csv', index=False)
bert_pred_df.head()


## Statistical comparison against the SVM baseline (McNemar's test)

Here we test whether DistilBERT's errors differ
systematically from the SVM baseline's errors on the same held-out examples.

In [ ]:
svm_pred_df = pd.read_csv('svm_test_predictions.csv')

assert len(svm_pred_df) == len(bert_pred_df), "Test set sizes differ between SVM and BERT runs!"
assert (svm_pred_df['y_true'].values == bert_pred_df['y_true'].values).all(), \
    "y_true mismatch -- check both notebooks used the same test_clean.csv without reshuffling"

svm_correct = (svm_pred_df['y_pred'].values == svm_pred_df['y_true'].values)
bert_correct = (bert_pred_df['y_pred'].values == bert_pred_df['y_true'].values)

both_correct = np.sum(svm_correct & bert_correct)
svm_only = np.sum(svm_correct & ~bert_correct)
bert_only = np.sum(~svm_correct & bert_correct)
both_wrong = np.sum(~svm_correct & ~bert_correct)

print("Contingency table:")
print(f"  Both correct:      {both_correct}")
print(f"  SVM only correct:  {svm_only}")
print(f"  BERT only correct: {bert_only}")
print(f"  Both wrong:        {both_wrong}")


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

table = [[both_correct, svm_only], [bert_only, both_wrong]]
result = mcnemar(table, exact=(svm_only + bert_only) < 25, correction=True)
print(f"McNemar's test statistic: {result.statistic:.4f}")
print(f"p-value: {result.pvalue:.6f}")

if result.pvalue < 0.05:
    print("=> Statistically significant difference between SVM and BERT at alpha=0.05")
else:
    print("=> No statistically significant difference between SVM and BERT at alpha=0.05")


## Bootstrap CI on macro-F1 for the representative run (test-set sampling uncertainty, separate from training-seed variance above)

In [ ]:
rng = np.random.RandomState(42)
n = len(test_labels_str_final)
n_boot = 2000
y_true_arr = np.array(test_labels_str_final)
y_pred_arr = np.array(bert_test_preds)

scores_f1 = []
for _ in range(n_boot):
    idx = rng.randint(0, n, n)
    scores_f1.append(f1_score(y_true_arr[idx], y_pred_arr[idx], average='macro', zero_division=0))

ci_lo, ci_hi = np.percentile(scores_f1, [2.5, 97.5])
print(f"Representative-seed macro-F1 bootstrap 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")


## Save final summary

In [ ]:
import json

summary = {
    'svm_baseline': {
        'test_accuracy': float(accuracy_score(svm_pred_df['y_true'], svm_pred_df['y_pred'])),
        'test_macro_f1': float(f1_score(svm_pred_df['y_true'], svm_pred_df['y_pred'], average='macro', zero_division=0)),
        'note': 'single run -- classical linear models have negligible training-run variance vs. neural nets, so multi-seed repetition was not run for this baseline.'
    },
    'distilbert_3seed': {
        'seeds': SEEDS,
        'per_seed_results': seed_results,
        'test_accuracy_mean': float(acc_mean),
        'test_accuracy_std': float(acc_std),
        'test_macro_f1_mean': float(f1_mean),
        'test_macro_f1_std': float(f1_std),
        'representative_seed_for_mcnemar': representative_seed,
        'representative_seed_bootstrap_macro_f1_ci': [float(ci_lo), float(ci_hi)],
    },
    'mcnemar_test_vs_svm': {
        'statistic': float(result.statistic),
        'p_value': float(result.pvalue),
        'contingency_table': {
            'both_correct': int(both_correct),
            'svm_only_correct': int(svm_only),
            'bert_only_correct': int(bert_only),
            'both_wrong': int(both_wrong),
        }
    }
}

with open('final_comparison_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


## Download results

Download these three files and bring them back -- the Results write-up will
be built from these real numbers.

In [ ]:
from google.colab import files
files.download('distilbert_seed_results.csv')
files.download('bert_test_predictions.csv')
files.download('final_comparison_summary.json')
